# **Approach 1:** Group products using existing fields

## Tasks 1 & 2: Match identical/variant products

In this approach, we assume that products with identical `brand`, `color`, `category`, `'name` and `description` are identical products, so we perform a grouping based on these attributes.

In [1]:
!pip install -qq torchvision
!pip install -qq hdbscan
!pip install -qq rapidfuzz

In [2]:
# Import libraries
import numpy as np
import pandas as pd
import json
import os
from collections import Counter, defaultdict
from itertools import count, product
from tqdm import tqdm

import matplotlib.pyplot as plt
from IPython.display import display, HTML

import requests
from io import BytesIO
from PIL import Image

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
import hdbscan
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder, normalize

from sklearn.metrics import silhouette_score, pair_confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances

import torch
import torchvision.models as models
import torchvision.transforms as transforms

### Function to visualize products

In [3]:
# Function to visualize clusters as HTML with an optional max_products limit
def show_clusters_as_html(df, cluster_ids, max_products=None):
    for cluster_id in cluster_ids:
        cluster_df = df[df['cluster_label'] == cluster_id]
        if cluster_df.empty:
            print(f"Cluster {cluster_id} is empty!")
            continue

        if max_products is not None:
            cluster_df = cluster_df.head(max_products)

        html = f"<h3 style='margin-top:20px'>Cluster {cluster_id} ({len(cluster_df)} items)</h3>"
        html += "<div style='display:flex;flex-wrap:wrap'>"

        for _, row in cluster_df.iterrows():
            name = row.get('product_name', '')
            
            # Choose the best available image
            for col in [
                'mainImage.original_url',
                'secondary_image_1.original_url',
                'secondary_image_2.original_url',
                'secondary_image_3.original_url'
            ]:
                img = row.get(col, '')
                if isinstance(img, str) and img.startswith('http'):
                    break
            else:
                img = ''
            
            html += f"""
                <div style='margin:10px;text-align:center'>
                    <img src='{img}' width='120' style='border-radius:8px'><br>
                    <div style='max-width:120px;font-size:12px'>{name}</div>
                </div>
            """
        html += "</div>"
        display(HTML(html))

# **EDA & preprocessing**

In [4]:
# Read data
df = pd.read_json('sneakers_sample_19062025.txt', lines=False)

print(df.head())

                                                data  \
0  {'EAN': [], 'size': '44', 'brand': 'ASICS', 'c...   
1  {'EAN': [], 'size': '44', 'brand': 'VANS', 'co...   
2  {'EAN': [], 'size': '44.5', 'brand': 'ADIDAS',...   
3  {'EAN': [], 'size': '42', 'brand': 'NIKE', 'co...   
4  {'EAN': [], 'size': '11/EU 46', 'brand': 'PUMA...   

                      mirakl_product_id             creation_date  \
0  d96975c5-0bea-41f6-a4a8-fdac53f374f3  2024-01-09T12:09:20.568Z   
1  6f302b78-9a86-484e-894f-bb39220931a3  2023-11-09T13:08:33.276Z   
2  70006973-aa9f-4176-9f38-0ba2c5a1b06f  2025-03-29T08:43:47.421Z   
3  6b010b1d-7475-44dc-ad56-502769d2e742  2024-01-04T12:29:59.343Z   
4  cf04d217-bfd6-4743-ae4b-bc98712959ca  2025-02-13T14:31:57.524Z   

                update_date                           product_sku  \
0  2024-11-06T13:03:01.469Z  d96975c5-0bea-41f6-a4a8-fdac53f374f3   
1  2025-02-12T12:21:44.083Z  6f302b78-9a86-484e-894f-bb39220931a3   
2  2025-03-30T21:07:23.464Z  70006973-aa9

In [5]:
df.head()

,data,mirakl_product_id,creation_date,update_date,product_sku,validation,synchronization,catalogs,product_urls,sources,selling_authorization,data_origin
0,"{'EAN': [], 'size': '44', 'brand': 'ASICS', 'c...",d96975c5-0bea-41f6-a4a8-fdac53f374f3,2024-01-09T12:09:20.568Z,2024-11-06T13:03:01.469Z,d96975c5-0bea-41f6-a4a8-fdac53f374f3,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2077', 'provider_sku': '10...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
1,"{'EAN': [], 'size': '44', 'brand': 'VANS', 'co...",6f302b78-9a86-484e-894f-bb39220931a3,2023-11-09T13:08:33.276Z,2025-02-12T12:21:44.083Z,6f302b78-9a86-484e-894f-bb39220931a3,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2002', 'provider_sku': 'VN...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
2,"{'EAN': [], 'size': '44.5', 'brand': 'ADIDAS',...",70006973-aa9f-4176-9f38-0ba2c5a1b06f,2025-03-29T08:43:47.421Z,2025-03-30T21:07:23.464Z,70006973-aa9f-4176-9f38-0ba2c5a1b06f,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2167', 'provider_sku': 'IH...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
3,"{'EAN': [], 'size': '42', 'brand': 'NIKE', 'co...",6b010b1d-7475-44dc-ad56-502769d2e742,2024-01-04T12:29:59.343Z,2024-11-06T13:00:53.470Z,6b010b1d-7475-44dc-ad56-502769d2e742,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2077', 'provider_sku': 'CT...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
4,"{'EAN': [], 'size': '11/EU 46', 'brand': 'PUMA...",cf04d217-bfd6-4743-ae4b-bc98712959ca,2025-02-13T14:31:57.524Z,2025-03-01T00:03:24.205Z,cf04d217-bfd6-4743-ae4b-bc98712959ca,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '3390', 'provider_sku': '46...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."


### Create separate dataframes

In [6]:
# Load the JSON file
with open('sneakers_sample_19062025.txt', 'r', encoding='utf-8') as f:    # added encoding = utf8 otherwise: decoding error
    raw_data = json.load(f)

# Create separate dataframes
meta_rows = []
data_rows = []
sources_rows = []

# Exclude some fields that are not necessary for our tasks
exclude_fields = {
    'creation_date', 'update_date',
    'validation', 'synchronization',
    'catalogs', 'selling_authorization', "product_urls"
}

for entry in raw_data:
    # Top-level fields
    meta = {
        key: entry[key] for key in entry
        if key not in exclude_fields and key not in ['data', 'sources']
    }
    meta_rows.append(meta)
    
    # Extract nested fields (if they exist)
    if 'data' in entry:
        data_rows.append(entry['data'])
    if 'sources' in entry and isinstance(entry['sources'], list):
        for source in entry['sources']:
            # Optionally, attach product ID to track back
            source['mirakl_product_id'] = entry.get('mirakl_product_id')
            sources_rows.append(source)

# Convert all parts to DataFrames
df_meta = pd.DataFrame(meta_rows)
df_data = pd.json_normalize(data_rows)
df_sources = pd.DataFrame(sources_rows)

# Preview
print("Meta columns:", df_meta.columns.tolist())
print("Data columns:", df_data.columns.tolist())
print("Sources sample:", df_sources.head())

Meta columns: ['mirakl_product_id', 'product_sku', 'data_origin']
Data columns: ['EAN', 'size', 'brand', 'color', 'category', 'returnable', 'name [el_CY]', 'name [en_GB]', 'updateOnDelta', 'refinementColor', 'variantGroupCode', 'jiniusSkuMatchCode', 'shortDescription [el_CY]', 'shortDescription [en_GB]', 'mainImage.source', 'mainImage.original_url', 'secondary_image_1.source', 'secondary_image_1.original_url', 'secondary_image_2.source', 'secondary_image_2.original_url', 'secondary_image_3.source', 'secondary_image_3.original_url', 'longDescription [en_GB]', 'longDescription [el_CY]', 'shoeType', 'shoeShape', 'fashionMaterial', 'shoeWidth', 'heelHeight', 'name', 'heelType', 'longDescription', 'shortDescription']
Sources sample:   provider_code                               provider_sku  \
0          2077        1071A091-101--EUM-18--WHITE / BLACK   
1          2002                   VN0A3MTJT2J1--417--BLACK   
2          2167                                IH6003-44.5   
3          207

In [7]:
df_data

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
0,[],44,ASICS,White / Black,FaMeShSneakers,true,Gel-rocket 11 mens volleyball/indoor shoes,Gel-rocket 11 mens volleyball/indoor shoes,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[],44,VANS,Black,FaMeShSneakers,true,Mens filmore suede,Mens filmore suede,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[],44.5,ADIDAS,NaN,FaMeShSneakers,true,Ih6003,Adidas response,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[],42,NIKE,White / Black / Volt,FaMeShSneakers,true,React vision worldwide mens,React vision worldwide mens,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[],11/EU 46,PUMA,Black/white,FaMeShSneakers,true,Puma men's x-ray 3 sd,Puma men's x-ray 3 sd,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5818,[],37,BUFFALO,White / Silver / Black,FaWoShSneakers,true,Triple hollow sneakers,Triple hollow sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5819,[],39,ADIDAS ORIGINALS,White,FaWoShSneakers,true,Adidas sambae w,Adidas sambae w,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5820,[],41,GUESS,White,FaWoShSneakers,true,Elbina sneakers,Elbina sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5821,[],36.5,NEW BALANCE,Cloud White / Burgundy,FaWoShSneakers,true,327 sportstyle sneakers,327 sportstyle sneakers,None,miscellaneous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_meta

,mirakl_product_id,product_sku,data_origin
0,d96975c5-0bea-41f6-a4a8-fdac53f374f3,d96975c5-0bea-41f6-a4a8-fdac53f374f3,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
1,6f302b78-9a86-484e-894f-bb39220931a3,6f302b78-9a86-484e-894f-bb39220931a3,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
2,70006973-aa9f-4176-9f38-0ba2c5a1b06f,70006973-aa9f-4176-9f38-0ba2c5a1b06f,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
3,6b010b1d-7475-44dc-ad56-502769d2e742,6b010b1d-7475-44dc-ad56-502769d2e742,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
4,cf04d217-bfd6-4743-ae4b-bc98712959ca,cf04d217-bfd6-4743-ae4b-bc98712959ca,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
...,...,...,...
5818,acdb8093-c93c-49d7-af05-f04cae009f99,acdb8093-c93c-49d7-af05-f04cae009f99,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
5819,e733c608-da84-49f1-a3ae-52a025b1cb3d,e733c608-da84-49f1-a3ae-52a025b1cb3d,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
5820,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
5821,e2d00db3-27b2-4996-89e2-5c1579af5d39,e2d00db3-27b2-4996-89e2-5c1579af5d39,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."


We observe that `data_origin` column is nested.

In [9]:
# Flatten 'data_origin' column
data_origin_df = pd.json_normalize(df_meta['data_origin'])

# Add prefixes so we know where they came from
data_origin_df = data_origin_df.add_prefix('origin_')

# Drop original nested columns from df_meta
df_meta = df_meta.drop(columns=['data_origin'])

# Merge the flattened parts back
df_meta = pd.concat([df_meta, data_origin_df], axis=1)

In [10]:
df_meta

,mirakl_product_id,product_sku,origin_name [en_GB].origin,origin_name [en_GB].provider_code,origin_name [el_CY].origin,origin_name [el_CY].provider_code,origin_color.origin,origin_color.provider_code,origin_shortDescription [el_CY].origin,origin_shortDescription [el_CY].provider_code,...,origin_heelHeight.origin,origin_heelHeight.provider_code,origin_longDescription.origin,origin_longDescription.provider_code,origin_shortDescription.origin,origin_shortDescription.provider_code,origin_name.origin,origin_name.provider_code,origin_heelType.origin,origin_heelType.provider_code
0,d96975c5-0bea-41f6-a4a8-fdac53f374f3,d96975c5-0bea-41f6-a4a8-fdac53f374f3,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6f302b78-9a86-484e-894f-bb39220931a3,6f302b78-9a86-484e-894f-bb39220931a3,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,70006973-aa9f-4176-9f38-0ba2c5a1b06f,70006973-aa9f-4176-9f38-0ba2c5a1b06f,PROVIDER,2167,PROVIDER,2167,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6b010b1d-7475-44dc-ad56-502769d2e742,6b010b1d-7475-44dc-ad56-502769d2e742,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cf04d217-bfd6-4743-ae4b-bc98712959ca,cf04d217-bfd6-4743-ae4b-bc98712959ca,PROVIDER,3390,PROVIDER,3390,PROVIDER,3390,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5818,acdb8093-c93c-49d7-af05-f04cae009f99,acdb8093-c93c-49d7-af05-f04cae009f99,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5819,e733c608-da84-49f1-a3ae-52a025b1cb3d,e733c608-da84-49f1-a3ae-52a025b1cb3d,PROVIDER,3429,PROVIDER,3429,PROVIDER,3429,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5820,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5821,e2d00db3-27b2-4996-89e2-5c1579af5d39,e2d00db3-27b2-4996-89e2-5c1579af5d39,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df_sources

,provider_code,provider_sku,mirakl_product_id
0,2077,1071A091-101--EUM-18--WHITE / BLACK,d96975c5-0bea-41f6-a4a8-fdac53f374f3
1,2002,VN0A3MTJT2J1--417--BLACK,6f302b78-9a86-484e-894f-bb39220931a3
2,2167,IH6003-44.5,70006973-aa9f-4176-9f38-0ba2c5a1b06f
3,2077,CT2927-100--USAM-09--WHITE / BLACK / VOLT,6b010b1d-7475-44dc-ad56-502769d2e742
4,3390,468781-1870785842,cf04d217-bfd6-4743-ae4b-bc98712959ca
...,...,...,...
5818,2077,1636155--EUM-05--WHITE / SILVER / BLACK,acdb8093-c93c-49d7-af05-f04cae009f99
5819,3429,JI1349--1310--WHITE,e733c608-da84-49f1-a3ae-52a025b1cb3d
5820,2002,FLJELBLEA12WHBEI--401--WHITE,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732
5821,2077,WS327-KA--EUM-04--CLOUD WHITE / BURGUNDY,e2d00db3-27b2-4996-89e2-5c1579af5d39


Our algorithms are **evaluated** on a manually labeled sample dataset. The labels correspond to variant groupings, which means that **only Task 2 can currently be evaluated**. However, Task 1 could be evaluated in a similar manner, provided we have a sample dataset labeled specifically for that task.

In [12]:
# Load the sample to evaluate the groups
sample_df = pd.read_csv("sample.csv")

Now that we have separate flattened dataframes we can investigate whether we have missing values and in which fields.

In [13]:
df_data.isna().sum()

EAN                                  0
size                                 0
brand                               11
color                               15
category                             0
returnable                           0
name [el_CY]                       608
name [en_GB]                         0
updateOnDelta                     5819
refinementColor                    733
variantGroupCode                     0
jiniusSkuMatchCode                   0
shortDescription [el_CY]          1336
shortDescription [en_GB]             0
mainImage.source                     0
mainImage.original_url               4
secondary_image_1.source           290
secondary_image_1.original_url     292
secondary_image_2.source           437
secondary_image_2.original_url     437
secondary_image_3.source           737
secondary_image_3.original_url     737
longDescription [en_GB]           5585
longDescription [el_CY]           5716
shoeType                          5681
shoeShape                

In [14]:
df_meta.isna().sum()

mirakl_product_id                                   0
product_sku                                         0
origin_name [en_GB].origin                          0
origin_name [en_GB].provider_code                 153
origin_name [el_CY].origin                        608
origin_name [el_CY].provider_code                 682
origin_color.origin                                15
origin_color.provider_code                        155
origin_shortDescription [el_CY].origin           1336
origin_shortDescription [el_CY].provider_code    1340
origin_secondary_image_2.origin                   437
origin_secondary_image_2.provider_code            446
origin_secondary_image_1.origin                   288
origin_secondary_image_1.provider_code            306
origin_shortDescription [en_GB].origin              0
origin_shortDescription [en_GB].provider_code      61
origin_secondary_image_3.origin                   737
origin_secondary_image_3.provider_code            741
origin_mainImage.origin     

In [15]:
df_sources.isna().sum()

provider_code        0
provider_sku         0
mirakl_product_id    0
dtype: int64

From the above we observe that some important fields (such as color, brand, descriptions) contain missing values.

In [16]:
# check the values of jiniusSkuMatchCode column
df_data["jiniusSkuMatchCode"].describe() 

count     5823
unique       1
top         []
freq      5823
Name: jiniusSkuMatchCode, dtype: object

`jiniusSkuMatchCode` column is only has empty lists, although it appeared to have no missing values.

In [17]:
#check the number of records with empty EAN
df_data["EAN"].describe()

count     5823
unique     487
top         []
freq      5337
Name: EAN, dtype: object

In [18]:
df_data["EAN"].apply(lambda x: x == []).sum()

5337

From the 5823 observations, 5337 of them have empty EAN code and the remaining have a unique EAN code. So no products are grouped by the EAN code.

In [19]:
df_data['brand'].isna().sum()

11

In [20]:
df_data[df_data['brand'].isna()][['name [en_GB]','name', 'color', 'category', 'shortDescription', 'brand']]

,name [en_GB],name,color,category,shortDescription,brand
1075,C8979,NaN,Black,FaMeShSneakers,NaN,NaN
2258,C8979,NaN,All White,FaMeShSneakers,NaN,NaN
2559,C8979,NaN,Black,FaMeShSneakers,NaN,NaN
2941,C895,NaN,White,FaWoShSneakers,NaN,NaN
3856,C895,NaN,Black,FaWoShSneakers,NaN,NaN
4193,C667,NaN,White,FaWoShSneakers,NaN,NaN
4469,C872,NaN,White,FaWoShSneakers,NaN,NaN
4777,C8962,NaN,Black,FaWoShSneakers,NaN,NaN
5033,C667,NaN,White,FaWoShSneakers,NaN,NaN
5451,C666,NaN,White,FaWoShSneakers,NaN,NaN


In [21]:
df_data[df_data['brand'] == 'CALVIN KLEIN JEANS']

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
148,[],45,CALVIN KLEIN JEANS,Black / White,FaMeShSneakers,true,Basket cupsole low leather sneakers,Basket cupsole low leather sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
335,[],45,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Retro tennis low laceup mtl,Retro tennis low laceup mtl,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
660,[],45,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Senakers,Senakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
675,[],42,CALVIN KLEIN JEANS,Black,FaMeShSneakers,true,Chunky cupsole authentic shoe,Chunky cupsole authentic shoe,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1204,[],43,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Chunky cupsole mono leather,Chunky cupsole mono leather,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1297,[],44,CALVIN KLEIN JEANS,Black,FaMeShSneakers,true,Retro tennis low laceup mtl,Retro tennis low laceup mtl,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1992,[],41,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Claic cupsole iconic nylon shoe,Claic cupsole iconic nylon shoe,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3580,[],40,CALVIN KLEIN JEANS,Black,FaWoShSneakers,true,Chunky cupsole laceup leather shoe,Chunky cupsole laceup leather shoe,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5088,[],40,CALVIN KLEIN JEANS,White,FaWoShSneakers,true,Retro tennis su-mesh wn,Retro tennis su-mesh wn,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df_data[df_data['brand'] == 'CALVIN KLEIN']

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
4411,[5299901986314],36,CALVIN KLEIN,White,FaWoShSneakers,true,Eva runner low lace shoes,Eva runner low lace shoes,None,white,...,NaN,0090,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The outputs above reveal inconsistencies in brand naming (e.g., *LIU JO SHOES* vs *LIU JO*). To ensure consistency, we apply a brand name mapping that standardizes all variations.

In [23]:
df_data['brand'].unique()

array(['ASICS', 'VANS', 'ADIDAS', 'NIKE', 'PUMA', 'CONVERSE',
       'ADIDAS ORIGINALS', 'UNDER ARMOUR', 'SAUCONY', 'REEBOK', 'JORDAN',
       'DIESEL', 'CHAMPION', 'NEW BALANCE', 'FRED PERRY', 'SKECHERS',
       'ON RUNNING', 'EA7', 'JOMA', 'CARRERA', "LEVI'S", 'VEJA',
       'PEPE SHOES', 'TALBOT SHOES', 'U.S. POLO ASSN.',
       'PEPE JEANS LONDON', 'TOMMY HILFIGER', 'CALVIN KLEIN JEANS', 'DC',
       'STEFAN', 'ANTONY MORATO', 'TOMMY JEANS', 'KAPPA', 'LYLE & SCOTT',
       'HACKETT', 'KARL KANI', 'SERGIO TACCHINI', 'TAMARIS', 'REVERSE',
       'BAZAAR CHARM', 'MORK EXCLUSIVE', 'LACOSTE', 'SOUTHPORT',
       'SALOMON', 'GAUDI', 'JACK & JONES', 'FUNKY BUDDHA', 'HUGO BOSS',
       'FENOMILANO', 'FILA', 'RHAPSODY', 'BOSTON SPORT CLUB', 'MIMSOGA',
       'SUN68', 'LIU JO SHOES', 'LOTTO', 'PLATO', 'MIZUNO', 'ADMIRAL',
       'BROOKS', 'SIKSILK', 'NICO', 'ICE PLAY', 'BHPC', 'GUESS DENIM',
       'PARIS', 'NO NAME ATHENS', 'GUESS', 'ATLANTA', nan,
       'BE YOU, U.S. POLO ASSN', 'THE NORT

In [24]:
brand_mapping = {
    'ADIDAS ORIGINALS': 'ADIDAS',
    'CALVIN KLEIN JEANS': 'CALVIN KLEIN',
    'LIU JO SHOES': 'LIU JO',
    'GUESS DENIM': 'GUESS',
    'TOMMY JEANS': 'TOMMY HILFIGER',
    'PEPE JEANS LONDON': 'PEPE JEANS',
    'BE YOU, U.S. POLO ASSN': 'U.S. POLO ASSN.',
    'BHPC': 'BEVERLY HILLS POLO CLUB',
    'EA7': 'EMPORIO ARMANI',
    'NO NAME ATHENS': 'NO NAME'
}

In [25]:
df_data_copy = df_data.copy()

df_data_copy['brand_label'] = df_data_copy['brand'].replace(brand_mapping)

In [26]:
df_data['brand'].nunique()

128

In [27]:
df_data_copy['brand'].nunique()

128

In [28]:
df_data_copy['brand_label'].nunique()

121

In [29]:
df_data_copy[df_data_copy['brand_label'] == 'ADIDAS']['name [en_GB]'].unique()

array(['Adidas response', 'Adidas gazelle shoes', 'Swift run 22',
       'Adistar 2 men', 'Midcity mid',
       "Adidas men's supernova stride shoes", 'Duramo speed men',
       'Adidas gazelle indoor', 'Gazelle shoes', 'Cloudfoam go lounge',
       'Midcity low', 'Strutter', 'Response runner unisex',
       'Response super mens', 'Adidas grand court base 2.0',
       'Stan smith vegan', 'Forest grove sneakers', 'Znsored hi mens',
       'Fukasa run', 'Cloudfoam comfy', 'Urban court',
       'Adidas alphaboost v1', 'Alphabounce + c', 'Samba classic',
       'Copa mundial', 'Ozelle', 'Adistar 3', 'X_plrphase',
       'Ultimashow 2.0',
       'Adidas questar             greone/cblack/grethr', 'Advantage 2.0',
       'Adidas copa pure.3 tf', 'Grand court 2.0', 'Samba og',
       'Adidas handball spezial', 'Handball spezial', 'Runfalcon 3.0',
       'Runfalcon 5', 'Adidas men duramo sl', 'Copa pure.1 firm ground',
       "Adidas men's duramo speed shoes", 'Park st',
       'Adidas men runf

## Task 1: Match identical products

#### **Approach 1:** Group products using existing fields

In this approach, we assume that products with identical `brand`, `color`, `category`, `'name` and `description` are identical products, so we perform a grouping based on these attributes.

In [30]:
# inspect features
print(df_data.columns)

Index(['EAN', 'size', 'brand', 'color', 'category', 'returnable',
       'name [el_CY]', 'name [en_GB]', 'updateOnDelta', 'refinementColor',
       'variantGroupCode', 'jiniusSkuMatchCode', 'shortDescription [el_CY]',
       'shortDescription [en_GB]', 'mainImage.source',
       'mainImage.original_url', 'secondary_image_1.source',
       'secondary_image_1.original_url', 'secondary_image_2.source',
       'secondary_image_2.original_url', 'secondary_image_3.source',
       'secondary_image_3.original_url', 'longDescription [en_GB]',
       'longDescription [el_CY]', 'shoeType', 'shoeShape', 'fashionMaterial',
       'shoeWidth', 'heelHeight', 'name', 'heelType', 'longDescription',
       'shortDescription'],
      dtype='object')


In [31]:
#column "name" is empty in almost all rows
df_data["name"].isna().sum()/df_data.shape[0]

0.9958784131890778

In [32]:
#find the only products with non-empty "name" column
df_data[df_data["name"].notna()]

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
1340,[198376576226],43,SKECHERS,Black,FaMeShSneakers,true,Skechers 118317 men shoes,Skechers 118317 men shoes,NaN,black,...,SKECHERS 118317 MEN SHOES / GLIDE STEP,0030,NaN,NaN,medium,NaN,SKECHERS 118317 MEN SHOES / GLIDE STEP,other,SKECHERS 118317 MEN SHOES / GLIDE STEP,SKECHERS 118317 MEN SHOES
1458,[198376147365],41,SKECHERS,Black,FaMeShSneakers,true,210890 bbk skechers men shoes,210890 bbk skechers men shoes,NaN,black,...,210890 BBK SKECHERS MEN SHOES/ SLIP IN,0030,NaN,NaN,medium,NaN,210890 BBK SKECHERS MEN SHOES,other,210890 BBK SKECHERS MEN SHOES/ SLIP IN,210890 BBK SKECHERS MEN SHOES
1805,[196989378503],42,SKECHERS,Grey,FaMeShSneakers,true,Skechers 232700 gry men shoes,Skechers 232700 grymen shoes,NaN,grey,...,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,0020,NaN,NaN,medium,NaN,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,other,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,SKECHERS 232700 GRY MEN SHOES
1853,[196989378527],43,SKECHERS,Grey,FaMeShSneakers,true,Skechers 232700 gry men shoes,Skechers 232700 grymen shoes,NaN,grey,...,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,0020,NaN,NaN,medium,NaN,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,other,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,SKECHERS 232700 GRY MEN SHOES
2443,[198376147419],43,SKECHERS,Black,FaMeShSneakers,true,210890 bbk skechers men shoes,210890 bbk skechers men shoes,NaN,black,...,210890 BBK SKECHERS MEN SHOES/ SLIP IN,0030,NaN,NaN,medium,NaN,210890 BBK SKECHERS MEN SHOES,other,210890 BBK SKECHERS MEN SHOES/ SLIP IN,210890 BBK SKECHERS MEN SHOES
3112,[197976864054],41,SKECHERS,Black,FaWoShSneakers,true,Skechers 150370 bbk women shoes,Skechers 150370 bbk women shoes,None,black,...,SKECHERS 150370 BBK WOMEN SHOES / AIR SOLE,0020,NaN,NaN,medium,NaN,SKECHERS 150370 BBK WOMEN SHOES,other,SKECHERS 150370 BBK WOMEN SHOES / AIR SOLE,SKECHERS 150370 BBK WOMEN SHOES
3455,[196989863429],38,SKECHERS,Black/gold,FaWoShSneakers,true,Skechers 117513 women shoes,Skechers 117513 women shoes,None,black,...,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 117513 WOMEN SHOES / BOBS SPORT,other,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,SKECHERS 117513 WOMEN SHOES
3457,[196989863405],37,SKECHERS,Black/gold,FaWoShSneakers,true,Skechers 117513 women shoes,Skechers 117513 women shoes,None,black,...,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 117513 WOMEN SHOES / BOBS SPORT,other,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,SKECHERS 117513 WOMEN SHOES
3498,[198376755850],39,SKECHERS,Beige/silver,FaWoShSneakers,true,Skechers 150246 ntsl women shoes,Skechers 150246 ntsl women shoes,NaN,beige,...,SKECHERS 150246 NTSL WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 150246 NTSL WOMEN SHOES,other,SKECHERS 150246 NTSL WOMEN SHOES / MEMORY FOAM,SKECHERS 150246 NTSL WOMEN SHOES
3550,[198739014495],41,SKECHERS,Black,FaWoShSneakers,true,Skechers 150562 bklv women shoes,Skechers 150562 bklv women shoes,NaN,black,...,SKECHERS 150562 BKLV WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 150562 BKLV WOMEN SHOES,other,SKECHERS 150562 BKLV WOMEN SHOES / MEMORY FOAM,SKECHERS 150562 BKLV WOMEN SHOES


In [33]:
#check if some name [el_CY] are missing
df_data["name [el_CY]"].isna().sum()

608

In [34]:
#check if some name [en_GB] are missing
df_data["name [en_GB]"].isna().sum() #no missing values

0

Given that `name [el_CY]` has missing values in some rows, while `name [en_GB]` not, we should only use the English name in the following grouping to provide more accurate results. 

A similar check for English and Greek descriptions.

In [35]:
#check if some shortDescription [el_CY] are missing
df_data["shortDescription [el_CY]"].isna().sum() 

1336

In [36]:
#check if some shortDescription [en_GB] are missing
df_data["shortDescription [en_GB]"].isna().sum() #no missing values

0

Again, some Greek descriptions are missing, while there are no missing English descriptions. So it would be better to use only English.

In [37]:
# may adjust these to be more/less strict!
group_keys = [     
    'brand',
    'color',   
    'category',
    'name [en_GB]',
    'shortDescription [en_GB]'] # provided that short descriptions are the same, there is no need to include long descriptions


# Group and count occurrences
group_sizes_en = df_data.groupby(group_keys).size().reset_index(name='count')

# Filter groups that occur more than once
duplicates_en = group_sizes_en[group_sizes_en['count'] > 1]

# Get full rows that belong to these duplicate groups
identical_products_en = df_data.merge(duplicates_en[group_keys], on=group_keys, how='inner')

# Assign group IDs
identical_products_en['product_group_id'] = identical_products_en.groupby(group_keys).ngroup()

# Create separate DataFrames for each group
grouped_dict_en = {
    f"group_{group_id}": group_df.drop(columns="product_group_id")
    for group_id, group_df in identical_products_en.groupby("product_group_id")
}

# Preview first 5 groups with only group keys + product_group_id
for group_id, df in list(grouped_dict_en.items())[:5]:
    print(f"\n===== Group: {group_id} =====")
    print(df[group_keys])


===== Group: group_0 =====
      brand      color        category name [en_GB]  \
43   ADIDAS  Aluminium  FaMeShSneakers       Ozelle   
127  ADIDAS  Aluminium  FaMeShSneakers       Ozelle   

                              shortDescription [en_GB]  
43   Regular fit<br>Lace closure<br>Textile upper<b...  
127  Regular fit<br>Lace closure<br>Textile upper<b...  

===== Group: group_1 =====
      brand  color        category name [en_GB]  \
89   ADIDAS  Beige  FaMeShSneakers   Adizero sl   
334  ADIDAS  Beige  FaMeShSneakers   Adizero sl   
581  ADIDAS  Beige  FaMeShSneakers   Adizero sl   

                              shortDescription [en_GB]  
89   <br>Lace closure for premium lockdown<br>Part ...  
334  <br>Lace closure for premium lockdown<br>Part ...  
581  <br>Lace closure for premium lockdown<br>Part ...  

===== Group: group_2 =====
      brand  color        category          name [en_GB]  \
489  ADIDAS  Black  FaMeShSneakers  Adidas men alphaedge   
692  ADIDAS  Black  FaMeSh

In [38]:
# add a column named 'task1_approach1' that has the group id for the current task and approach
df_data['task1_approach1'] = df_data.groupby(group_keys).ngroup()
df_data

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription,task1_approach1
0,[],44,ASICS,White / Black,FaMeShSneakers,true,Gel-rocket 11 mens volleyball/indoor shoes,Gel-rocket 11 mens volleyball/indoor shoes,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1041.0
1,[],44,VANS,Black,FaMeShSneakers,true,Mens filmore suede,Mens filmore suede,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4423.0
2,[],44.5,ADIDAS,NaN,FaMeShSneakers,true,Ih6003,Adidas response,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[],42,NIKE,White / Black / Volt,FaMeShSneakers,true,React vision worldwide mens,React vision worldwide mens,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2839.0
4,[],11/EU 46,PUMA,Black/white,FaMeShSneakers,true,Puma men's x-ray 3 sd,Puma men's x-ray 3 sd,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3168.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5818,[],37,BUFFALO,White / Silver / Black,FaWoShSneakers,true,Triple hollow sneakers,Triple hollow sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1132.0
5819,[],39,ADIDAS ORIGINALS,White,FaWoShSneakers,true,Adidas sambae w,Adidas sambae w,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,882.0
5820,[],41,GUESS,White,FaWoShSneakers,true,Elbina sneakers,Elbina sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1589.0
5821,[],36.5,NEW BALANCE,Cloud White / Burgundy,FaWoShSneakers,true,327 sportstyle sneakers,327 sportstyle sneakers,None,miscellaneous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2028.0


In [39]:
# count products with empty group ids
df_data['task1_approach1'].isna().sum()  

26

From the above output we observe that some products have not been grouped using Approach 1. Further exploration of these products:

In [40]:
#explore the products that have not been grouped
df_data[df_data['task1_approach1'].isna()].head()

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription,task1_approach1
2,[],44.5,ADIDAS,NaN,FaMeShSneakers,true,Ih6003,Adidas response,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123,[],42,TALBOT SHOES,NaN,FaMeShSneakers,true,Joma men,Joma men,None,NaN,...,0090,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
402,[],46,TALBOT SHOES,NaN,FaMeShSneakers,true,Joma men,Joma men,None,NaN,...,0090,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
605,[],45.5,ADIDAS,NaN,FaMeShSneakers,true,Id8760,Adidas galaxy 7,NaN,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
868,[],42.5,ON RUNNING,NaN,FaMeShSneakers,true,On running cloud 5,On running cloud 5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


From the above output we observe that, even though the products with indices 123 and 402 are the same, they have not been grouped together. I suspect that this might be due to having missing colors and Python might treat them as not having the same color. Let's investigate this assumption:

In [41]:
df_data.iloc[123,3] == df_data.iloc[402,3]

False

So the above verifies that the above assumption is true. Following, we group without considering colors to capture variants **(Task 2)**.

In [42]:
# Check number of grouped created
df_data["task1_approach1"].nunique()

4608

In [43]:
df_data.shape

(5823, 34)

Out of the 5823 records, **approach 1** created 4608 groups. This means that we grouped some products that are identical w.r.t. the previous fields. However very few groups were created. 

## Task 2: Match similar products with different variants

#### **Approach 1:** Group products using existing fields

The idea is similar to that of **Task 1**. Here we remove `color` from group keys. This results in groups of identical products in the same/ different colors.

In [44]:
# may adjust these to be more/less strict!
group_keys = [
    'brand',
    'category',
    'name [en_GB]',
    'shortDescription [en_GB]']

# Group and count occurrences
group_sizes_en = df_data.groupby(group_keys).size().reset_index(name='count')

# Filter groups that occur more than once
duplicates_en = group_sizes_en[group_sizes_en['count'] > 1]

# Get full rows that belong to these duplicate groups
identical_products_en = df_data.merge(duplicates_en[group_keys], on=group_keys, how='inner')

# Assign group IDs
identical_products_en['product_group_id'] = identical_products_en.groupby(group_keys).ngroup()

# Create separate DataFrames for each group
grouped_dict_en = {
    f"group_{group_id}": group_df.drop(columns="product_group_id")
    for group_id, group_df in identical_products_en.groupby("product_group_id")
}

# Preview first 5 groups with only group keys + product_group_id
for group_id, df in list(grouped_dict_en.items())[:5]:
    print(f"\n===== Group: {group_id} =====")
    print(df[group_keys])


===== Group: group_0 =====
      brand        category name [en_GB]  \
233  ADIDAS  FaMeShSneakers  4dfwd 2 men   
516  ADIDAS  FaMeShSneakers  4dfwd 2 men   

                              shortDescription [en_GB]  
233  <br>Lace closure<br>adidas PRIMEKNIT textile u...  
516  <br>Lace closure<br>adidas PRIMEKNIT textile u...  

===== Group: group_1 =====
      brand        category  name [en_GB]  \
179  ADIDAS  FaMeShSneakers  4dfwd 2 mens   
521  ADIDAS  FaMeShSneakers  4dfwd 2 mens   

                              shortDescription [en_GB]  
179  <br>Lace closure<br>adidas PRIMEKNIT textile u...  
521  <br>Lace closure<br>adidas PRIMEKNIT textile u...  

===== Group: group_2 =====
       brand        category name [en_GB] shortDescription [en_GB]
207   ADIDAS  FaMeShSneakers  4dfwd 4 men              4DFWD 4 Men
312   ADIDAS  FaMeShSneakers  4dfwd 4 men              4DFWD 4 Men
866   ADIDAS  FaMeShSneakers  4dfwd 4 men              4DFWD 4 Men
1246  ADIDAS  FaMeShSneakers  4dfwd 4

In [45]:
# add a column with the group id
df_data['task2_approach1'] = df_data.groupby(group_keys).ngroup()
df_data

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription,task1_approach1,task2_approach1
0,[],44,ASICS,White / Black,FaMeShSneakers,true,Gel-rocket 11 mens volleyball/indoor shoes,Gel-rocket 11 mens volleyball/indoor shoes,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1041.0,843.0
1,[],44,VANS,Black,FaMeShSneakers,true,Mens filmore suede,Mens filmore suede,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4423.0,3737.0
2,[],44.5,ADIDAS,NaN,FaMeShSneakers,true,Ih6003,Adidas response,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,107.0
3,[],42,NIKE,White / Black / Volt,FaMeShSneakers,true,React vision worldwide mens,React vision worldwide mens,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2839.0,2099.0
4,[],11/EU 46,PUMA,Black/white,FaMeShSneakers,true,Puma men's x-ray 3 sd,Puma men's x-ray 3 sd,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3168.0,2609.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5818,[],37,BUFFALO,White / Silver / Black,FaWoShSneakers,true,Triple hollow sneakers,Triple hollow sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1132.0,933.0
5819,[],39,ADIDAS ORIGINALS,White,FaWoShSneakers,true,Adidas sambae w,Adidas sambae w,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,882.0,681.0
5820,[],41,GUESS,White,FaWoShSneakers,true,Elbina sneakers,Elbina sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1589.0,1332.0
5821,[],36.5,NEW BALANCE,Cloud White / Burgundy,FaWoShSneakers,true,327 sportstyle sneakers,327 sportstyle sneakers,None,miscellaneous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2028.0,1755.0


In [46]:
# investigate for products with missing group
df_data['task2_approach1'].isna().sum() 

11

Again, some products were not grouped. Further investigate them:

In [47]:
#explore the products that have not been grouped
df_data[df_data['task2_approach1'].isna()].head()

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription,task1_approach1,task2_approach1
1075,[],45,NaN,Black,FaMeShSneakers,true,C8979,C8979,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2258,[],43,NaN,All White,FaMeShSneakers,true,C8979,C8979,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2559,[],40,NaN,Black,FaMeShSneakers,true,C8979,C8979,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2941,[],39,NaN,White,FaWoShSneakers,true,C895,C895,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3856,[],36,NaN,Black,FaWoShSneakers,true,C895,C895,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# Check number of grouped created
df_data["task2_approach1"].nunique()

3900

From the above output we observe that products 1075, 2258, 2559 are the same (with different color) but were not grouped. The issue is that brand is missing so they could not be grouped.  
**Problem that we observe with Approach 1 (in both tasks):** Some products, despite being identical/variants, fail to be grouped. This is because some fields (that are used in grouping) contain missing values. However this corresponds to only a few cases and it can be fixed manually, or we could be less strict and group only based on fields with no missing values.   

The following approaches reveal that additional products can be grouped together. Therefore, we do not proceed with further exploration of approach 1.

**Approach 1** is the naive approach and doesn't require any evaluation: products that were grouped are indeed identical (task 1) or variants (task 2). Following, we explore some other approaches including image and language processing.